In [3]:
print("hello world")

hello world


In [5]:
print("hello world")
import datetime as dt
import pandas as pd
from nselib import capital_market as cm
from nselib import derivatives as der
from nselib import indices as idx


hello world


In [6]:
import pandas as pd
from nselib import capital_market

# Fetch all NSE listed companies
df = capital_market.equity_list()

# List all columns in df
print(df.columns.tolist())

print(df)


# Select useful columns
df = df[[
    "SYMBOL",
    "NAME OF COMPANY"

]]

# Clean column names
df.columns = ["symbol", "company"]

# Show first 20 rows
print(df.head(20))

# Save to CSV (optional)
df.to_csv("nse_companies_marketcap.csv", index=False)


['SYMBOL', 'NAME OF COMPANY', ' SERIES', ' DATE OF LISTING', ' FACE VALUE']
          SYMBOL                           NAME OF COMPANY  SERIES  \
0      20MICRONS                        20 Microns Limited      EQ   
1     21STCENMGM  21st Century Management Services Limited      EQ   
2         360ONE                       360 ONE WAM LIMITED      EQ   
3     3BBLACKBIO                    3B Blackbio Dx Limited      EQ   
4      3IINFOLTD                       3i Infotech Limited      BE   
...          ...                                       ...     ...   
2563    ZSARACOM        Saraswati Commercial India Limited      EQ   
2564       ZUARI              Zuari Agro Chemicals Limited      EQ   
2565    ZUARIIND                  ZUARI INDUSTRIES LIMITED      EQ   
2566   ZYDUSLIFE                Zydus Lifesciences Limited      EQ   
2567   ZYDUSWELL                    Zydus Wellness Limited      EQ   

      DATE OF LISTING   FACE VALUE  
0         06-OCT-2008            5  
1        

In [7]:
import pandas as pd
from nselib import capital_market

def fetch_tcs(period="6M"):
    """
    Fetch historical NSE data for TCS using nselib.
    period options: 1M, 3M, 6M, 1Y, 2Y, 5Y
    """
    df = capital_market.price_volume_and_deliverable_position_data(
        symbol="TCS",
        period=period
    )

    # Standardize column names
    df.rename(columns={
        "Date": "date",
        "Open Price": "open",
        "High Price": "high",
        "Low Price": "low",
        "Close Price": "close",
        "Total Traded Quantity": "volume"
    }, inplace=True)

    df["date"] = pd.to_datetime(df["date"])
    df.sort_values("date", inplace=True)

    return df


def compute_macd(df):
    """
    Compute MACD (12, 26, 9) manually using pandas.
    """

    # EMA calculations
    df["EMA12"] = df["close"].ewm(span=12, adjust=False).mean()
    df["EMA26"] = df["close"].ewm(span=26, adjust=False).mean()

    # MACD Line
    df["MACD"] = df["EMA12"] - df["EMA26"]

    # Signal Line (9‑period EMA of MACD)
    df["Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()

    # Histogram
    df["Hist"] = df["MACD"] - df["Signal"]

    return df


def detect_macd_cross(df):
    """
    Detect bullish/bearish MACD crossovers.
    """

    df["bullish_cross"] = (
        (df["MACD"] > df["Signal"]) &
        (df["MACD"].shift(1) < df["Signal"].shift(1))
    )

    df["bearish_cross"] = (
        (df["MACD"] < df["Signal"]) &
        (df["MACD"].shift(1) > df["Signal"].shift(1))
    )

    latest = df.iloc[-1]

    if latest["bullish_cross"]:
        return "Bullish MACD crossover"
    elif latest["bearish_cross"]:
        return "Bearish MACD crossover"
    else:
        return "No fresh crossover"


def run_macd_scan():
    df = fetch_tcs(period="6M")
    df = compute_macd(df)
    signal = detect_macd_cross(df)

    print("\n=== TCS (NSE) MACD Signal ===")
    print(signal)
    print("\nLatest MACD values:")
    print(df[["date", "MACD", "Signal", "Hist"]].tail())


# Run
run_macd_scan()


KeyError: 'close'

In [10]:
df = capital_market.price_volume_and_deliverable_position_data(
symbol="TCS",
period="6M")

In [ ]:
print(df.columns.tolist())